# HyperKT Stage 9D — Performance Gap and Fairness Audit
**Runtime:** CPU. **Inputs:** `hyperkt_stage8b.zip`, `hyperkt_stage9c.zip`; optionally recovered Stage 8B checkpoint ZIP.  
**Purpose:** Quantify the validation gap between DKT and HyperKT, examine paired seed differences and training fairness, and document a controlled improvement plan. **This notebook does not train models or evaluate the held-out test set.**

Statistical intervals are exploratory because seeds share the same validation cohort and model families differ. Do not claim an independent-subject significance test.

In [ ]:
%pip -q install numpy pandas scipy matplotlib

In [ ]:
import io, json, zipfile, hashlib, platform, os
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
OUT=Path('/content/hyperkt_stage9d') if Path('/content').exists() else Path.cwd()/'hyperkt_stage9d'
OUT.mkdir(parents=True,exist_ok=True)
try:
 from google.colab import files
 uploaded=files.upload()
 paths={name:Path('/content')/name for name in uploaded}
except ImportError:
 paths={p.name:p for p in Path('/mnt/data').glob('*.zip')}
if not paths: paths={p.name:p for p in Path.cwd().glob('*.zip')}
def locate(prefix):
 hits=[p for name,p in paths.items() if name.lower().startswith(prefix) and name.lower().endswith('.zip')]
 if not hits: raise FileNotFoundError(f'Upload {prefix}*.zip')
 return hits[0]
p8,p9=locate('hyperkt_stage8b'),locate('hyperkt_stage9c')
print('Stage 8B:',p8,'Stage 9C:',p9)

In [ ]:
def read_archive(path,filename):
 with zipfile.ZipFile(path) as z:
  return z.read(filename)
a=pd.read_csv(io.BytesIO(read_archive(p8,'all_results.csv')))
b=pd.read_csv(io.BytesIO(read_archive(p9,'all_results.csv')))
c8=json.loads(read_archive(p8,'config.json'))
c9=json.loads(read_archive(p9,'protocol.json'))
assert len(a)==20 and len(b)==20
assert not a.duplicated(['seed','variant']).any() and not b.duplicated(['seed','variant']).any()
assert set(a.seed)==set(b.seed)=={42,123,2024,3407,7777}
assert a.n.eq(23858).all() and b.n_targets.eq(23858).all()
assert c8['stage2_sha256']==c9['stage2_zip_sha256'], 'Dataset ZIP hashes differ'
print('Matching dataset hashes; 20 + 20 records, five common seeds')
print('HyperKT settings:',c8['config']);print('Baseline settings:',c9['config'])

## 1. Recalculate all results and paired seed differences
The paired t interval below summarizes **variation across the five training seeds**, not uncertainty across independent students. The Wilcoxon test is exploratory with only five observations.

In [ ]:
full=a[a.variant=='transformer_hypergraph_decay'].set_index('seed').sort_index()
dkt=b[b.variant=='DKT'].set_index('seed').sort_index()
comparison=pd.DataFrame({'seed':full.index,'HyperKT_auc':full.auc.values,'DKT_auc':dkt.auc.values,'DKT_minus_HyperKT':dkt.auc.values-full.auc.values,
 'HyperKT_nll':full.nll.values,'DKT_nll':dkt.nll.values,'HyperKT_brier':full.brier.values,'DKT_brier':dkt.brier.values})
comparison.to_csv(OUT/'paired_seed_comparison.csv',index=False)
diff=comparison['DKT_minus_HyperKT'].to_numpy();mean=diff.mean();sd=diff.std(ddof=1)
ci=stats.t.interval(.95,df=len(diff)-1,loc=mean,scale=stats.sem(diff))
t=stats.ttest_1samp(diff,0)
try: w=stats.wilcoxon(diff).pvalue
except ValueError: w=np.nan
stat_report={'n_seeds':len(diff),'mean_DKT_minus_HyperKT_auc':float(mean),'sd':float(sd),'ci95_t':[float(ci[0]),float(ci[1])],
 'exploratory_paired_t_p':float(t.pvalue),'exploratory_wilcoxon_p':float(w),'positive_seeds':int((diff>0).sum()),'shared_validation_cohort':True}
(OUT/'statistical_report.json').write_text(json.dumps(stat_report,indent=2))
print(comparison.to_string(index=False));print(json.dumps(stat_report,indent=2))

In [ ]:
variants=pd.concat([a.assign(source='8B'),b.assign(source='9C')],ignore_index=True)
summary=variants.groupby(['source','variant']).agg(auc_mean=('auc','mean'),auc_sd=('auc','std'),nll_mean=('nll','mean'),brier_mean=('brier','mean'),f1_mean=('f1','mean'),ece_mean=('ece15','mean')).reset_index()
summary.to_csv(OUT/'all_model_summary.csv',index=False)
print(summary.sort_values('auc_mean',ascending=False).to_string(index=False))
fig,ax=plt.subplots(figsize=(9,4))
sub=summary.sort_values('auc_mean')
ax.barh(sub['variant'],sub['auc_mean'],xerr=sub['auc_sd'],capsize=3)
ax.set_xlim(.72,.78);ax.set_xlabel('Validation ROC-AUC (mean ± seed SD)')
ax.set_title('Stage 9D: descriptive five-seed comparison');fig.tight_layout();fig.savefig(OUT/'validation_comparison.png',dpi=180);plt.show()

## 2. Fairness audit
A shared dataset and five common seed labels do **not** imply matched capacity or matched optimization. The baseline and HyperKT experiments used different batch sizes (32 vs 16). The implementations also differ in model families, and AKT is only AKT-inspired.

We report trainable parameter counts from Stage 9C records. For HyperKT, the original Stage 8B source code is used to calculate the parameter count without training or evaluating data.

In [ ]:
import nbformat, ast
src_candidates=[Path('/content/HyperKT_Stage8B_Five_Seed_Colab.ipynb'),Path('/mnt/data/HyperKT_Stage8B_Five_Seed_Colab.ipynb')]
source=next((p for p in src_candidates if p.exists()),None)
print('Stage 8B notebook available:',bool(source))
print('Stage 9C recorded trainable parameter counts:')
print(b.groupby('variant').n_parameters.agg(['min','max']).to_string())
if source:
 nb=nbformat.read(source,as_version=4)
 for i,c in enumerate(nb.cells):
  if 'class HyperKT(' in c.source:
   print('HyperKT class definition found in cell',i,'(count requires executing original model class with its dependencies)')
   break
else: print('Optional: upload original Stage 8B notebook to inspect HyperKT parameter count.')
fairness={'shared_dataset_zip_sha256':c8['stage2_sha256'],'same_validation_targets':True,'same_seed_labels':True,
 'stage8b_batch_size':c8['config']['batch_size'],'stage9c_batch_size':c9['config']['batch_size'],
 'same_batch_size':c8['config']['batch_size']==c9['config']['batch_size'],
 'same_max_epochs':c8['config']['max_epochs']==c9['config']['max_epochs'],
 'same_patience':c8['config']['patience']==c9['config']['patience'],
 'baseline_implementation':c9['baseline_status'],
 'cross_window_event_uniqueness':c9['known_limitation'],
 'hypergraph_fixed_across_seeds':True,'test_evaluated':False}
(OUT/'fairness_audit.json').write_text(json.dumps(fairness,indent=2))
print(json.dumps(fairness,indent=2))

## 3. Controlled improvement protocol (proposal only)
**Do not relabel the existing Stage 8B results as outcomes of this new protocol.** If improvements are attempted, label them Stage 9E and retain the current model as the locked historical reference.

1. **Architecture/capacity:** calculate exact trainable parameter counts and compare under matched or clearly reported budgets. Use the same training batches and optimizer schedule for the controlled retraining experiment.
2. **Fusion ablations:** compare fixed additive fusion, learned gated fusion, and a no-hypergraph control. Avoid assuming a benefit.
3. **Structural source:** compare learned Stage 3B hypergraph with randomized/permuted hypergraph embeddings and degree-controlled alternatives to detect whether structure itself matters.
4. **Optimization:** evaluate hypergraph freezing vs training only if permitted by the predefined validation search; document all additional searches and seeds.
5. **Dataset integrity:** recover original interaction event IDs to resolve cross-window event uniqueness. If not possible, document the limitation.
6. **Baselines:** use official or carefully validated reference implementations where feasible; AKT-inspired is not canonical AKT.
7. **Final test:** do not evaluate until the protocol and model-selection criteria are frozen. All results here are validation only.

**Suggested decision rule:** keep the DKT baseline as the strongest observed validation comparator; only select a modified HyperKT after a predeclared, repeated-seed validation experiment and a fairness audit. Do not optimize to a predetermined target AUC.

In [ ]:
protocol={
 'stage':'9D','decision':'DIAGNOSTIC_ONLY_NOT_TEST_READY',
 'best_observed_baseline':'DKT','candidate':'transformer_hypergraph_decay',
 'auc_gap_DKT_minus_HyperKT':float(mean),
 'blockers':['unequal batch sizes and architecture capacity not fully matched','official published baseline fidelity not established',
 'cross-window event uniqueness unresolved','any proposed improvement requires a separately labeled retraining study'],
 'next':'Stage 9E controlled improvement and fairness experiments, then protocol freeze and held-out test'}
(OUT/'protocol_recommendation.json').write_text(json.dumps(protocol,indent=2))
import zipfile
archive=Path('/content/hyperkt_stage9d.zip') if Path('/content').exists() else Path.cwd()/'hyperkt_stage9d.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for f in OUT.iterdir():
  if f.is_file():z.write(f,f.name)
print('Created',archive,'size',archive.stat().st_size)
try:
 from google.colab import files
 files.download(str(archive))
except ImportError:pass